In [1]:
# Import required libraries
import json
import pathlib
import numpy as np
import torch
import h5py
from PIL import Image
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple

# ARTIST imports
from artist.scenario.scenario import Scenario
from artist.core.heliostat_ray_tracer import HeliostatRayTracer
from artist.util import config_dictionary, set_logger_config, index_mapping
from artist.util.environment_setup import get_device

# Set up logger
set_logger_config()

# Set device
device = get_device()
print(f"Using device: {device}")

[2025-12-15 16:24:33,627][artist.util.environment_setup][INFO] - No device type provided. The device will default to GPU based on availability and OS, otherwise to CPU.
[2025-12-15 16:24:33,627][artist.util.environment_setup][WARNING] - Setting device to CPU. ARTIST only supports CPU for MacOS.
Using device: cpu


In [2]:
# Define paths to data
DATA_DIR = pathlib.Path("tutorials/data/paint")
TOWER_FILE = DATA_DIR / "tower-measurements.json"

# Define the 3 calibration samples targeting solar_tower_juelich_lower
# We'll use all 3 heliostats to get 3 samples total (for train-test split)
CALIBRATION_SAMPLES = [
    {
        'heliostat_id': 'AA31',
        'calibration_id': '126372',
        'heliostat_props': DATA_DIR / 'AA31' / 'heliostat-properties.json',
        'calibration_file': DATA_DIR / 'AA31' / '126372-calibration-properties.json',
        'flux_image': DATA_DIR / 'AA31' / '126372-flux.png',
        'deflectometry': DATA_DIR / 'AA31' / 'deflectometry.h5',
    },
    {
        'heliostat_id': 'AA39',
        'calibration_id': '271633',
        'heliostat_props': DATA_DIR / 'AA39' / 'heliostat-properties.json',
        'calibration_file': DATA_DIR / 'AA39' / '271633-calibration-properties.json',
        'flux_image': DATA_DIR / 'AA39' / '271633-flux.png',
        'deflectometry': DATA_DIR / 'AA39' / 'deflectometry.h5',
    },
    {
        'heliostat_id': 'AC43',
        'calibration_id': '72752',
        'heliostat_props': DATA_DIR / 'AC43' / 'heliostat-properties.json',
        'calibration_file': DATA_DIR / 'AC43' / '72752-calibration-properties.json',
        'flux_image': DATA_DIR / 'AC43' / '72752-flux.png',
        'deflectometry': DATA_DIR / 'AC43' / 'deflectometry.h5',
    },
]

print(f"Total calibration samples: {len(CALIBRATION_SAMPLES)}")
print("Samples:")
for sample in CALIBRATION_SAMPLES:
    print(f"  - {sample['heliostat_id']}/{sample['calibration_id']}")

Total calibration samples: 3
Samples:
  - AA31/126372
  - AA39/271633
  - AC43/72752


In [3]:
# GPS to ENU conversion function
def gps_to_enu(lat, lon, alt, origin_lat, origin_lon, origin_alt):
    """
    Convert GPS (WGS84) coordinates to local ENU (East-North-Up) coordinates.
    
    Args:
        lat, lon, alt: Target GPS coordinates
        origin_lat, origin_lon, origin_alt: Origin GPS coordinates
        
    Returns:
        numpy array [E, N, U] in meters
    """
    # Convert degrees to radians
    lat_rad = np.radians(lat)
    lon_rad = np.radians(lon)
    origin_lat_rad = np.radians(origin_lat)
    origin_lon_rad = np.radians(origin_lon)
    
    # WGS84 ellipsoid parameters
    a = 6378137.0  # semi-major axis in meters
    f = 1 / 298.257223563  # flattening
    e2 = 2 * f - f ** 2  # first eccentricity squared
    
    # Radius of curvature in the prime vertical
    N_origin = a / np.sqrt(1 - e2 * np.sin(origin_lat_rad) ** 2)
    N_target = a / np.sqrt(1 - e2 * np.sin(lat_rad) ** 2)
    
    # Convert to ECEF (Earth-Centered Earth-Fixed)
    X_origin = (N_origin + origin_alt) * np.cos(origin_lat_rad) * np.cos(origin_lon_rad)
    Y_origin = (N_origin + origin_alt) * np.cos(origin_lat_rad) * np.sin(origin_lon_rad)
    Z_origin = (N_origin * (1 - e2) + origin_alt) * np.sin(origin_lat_rad)
    
    X_target = (N_target + alt) * np.cos(lat_rad) * np.cos(lon_rad)
    Y_target = (N_target + alt) * np.cos(lat_rad) * np.sin(lon_rad)
    Z_target = (N_target * (1 - e2) + alt) * np.sin(lat_rad)
    
    # Difference in ECEF
    dX = X_target - X_origin
    dY = Y_target - Y_origin
    dZ = Z_target - Z_origin
    
    # Rotation matrix from ECEF to ENU
    E = -np.sin(origin_lon_rad) * dX + np.cos(origin_lon_rad) * dY
    N = -np.sin(origin_lat_rad) * np.cos(origin_lon_rad) * dX - np.sin(origin_lat_rad) * np.sin(origin_lon_rad) * dY + np.cos(origin_lat_rad) * dZ
    U = np.cos(origin_lat_rad) * np.cos(origin_lon_rad) * dX + np.cos(origin_lat_rad) * np.sin(origin_lon_rad) * dY + np.sin(origin_lat_rad) * dZ
    
    return np.array([E, N, U])

# Test the function
print("GPS to ENU conversion function defined")

GPS to ENU conversion function defined


In [4]:
# Load tower measurements to get power plant origin and target positions
with open(TOWER_FILE) as f:
    tower_data = json.load(f)

# Extract power plant GPS coordinates (will be our origin)
power_plant_gps = tower_data['power_plant_properties']['coordinates']
origin_lat, origin_lon, origin_alt = power_plant_gps

print(f"Power plant position (origin): lat={origin_lat}, lon={origin_lon}, alt={origin_alt}m")

# Extract target solar_tower_juelich_lower center position and convert to ENU
target_center_gps = tower_data['solar_tower_juelich_lower']['coordinates']['center']
target_lat, target_lon, target_alt = target_center_gps

target_enu = gps_to_enu(target_lat, target_lon, target_alt, origin_lat, origin_lon, origin_alt)

print(f"Target (solar_tower_juelich_lower) GPS: lat={target_lat}, lon={target_lon}, alt={target_alt}m")
print(f"Target ENU position: E={target_enu[0]:.2f}, N={target_enu[1]:.2f}, U={target_enu[2]:.2f}")

Power plant position (origin): lat=50.9134211225926, lon=6.38782475587486, alt=87m
Target (solar_tower_juelich_lower) GPS: lat=50.91339203684, lon=6.38782456351324, alt=122.8815m
Target ENU position: E=-0.01, N=-3.24, U=35.88


In [5]:
# Load all calibration samples
def load_sample_data(sample_info):
    """Load all data for one calibration sample."""
    # Load calibration properties (sun angles, motor positions)
    with open(sample_info['calibration_file']) as f:
        calib_data = json.load(f)
    
    # Load heliostat properties (GPS position)
    with open(sample_info['heliostat_props']) as f:
        heliostat_data = json.load(f)
    
    # Load flux image
    flux_img = Image.open(sample_info['flux_image']).convert('L')
    flux_array = np.array(flux_img, dtype=np.float32) / 255.0  # Normalize to [0, 1]
    
    # Extract data
    sun_elevation = calib_data['sun_elevation']
    sun_azimuth = calib_data['sun_azimuth']
    motor_1 = calib_data['motor_position']['axis_1_motor_position']
    motor_2 = calib_data['motor_position']['axis_2_motor_position']
    
    # Get heliostat GPS position and convert to ENU
    heliostat_gps = heliostat_data['heliostat_position']  # Directly at top level
    heliostat_enu = gps_to_enu(
        heliostat_gps[0], heliostat_gps[1], heliostat_gps[2],
        origin_lat, origin_lon, origin_alt
    )
    
    return {
        'heliostat_id': sample_info['heliostat_id'],
        'calibration_id': sample_info['calibration_id'],
        'sun_elevation': sun_elevation,
        'sun_azimuth': sun_azimuth,
        'heliostat_enu': heliostat_enu,
        'tower_enu': np.array([0.0, 0.0, 0.0]),  # Origin
        'target_enu': target_enu,
        'motor_1': motor_1,
        'motor_2': motor_2,
        'flux_image': flux_array,
    }

# Load all samples
print("Loading all calibration samples...")
all_samples = [load_sample_data(sample_info) for sample_info in CALIBRATION_SAMPLES]

print(f"\nLoaded {len(all_samples)} samples:")
for i, sample in enumerate(all_samples):
    print(f"\nSample {i}: {sample['heliostat_id']}/{sample['calibration_id']}")
    print(f"  Sun: elevation={sample['sun_elevation']:.2f}°, azimuth={sample['sun_azimuth']:.2f}°")
    print(f"  Heliostat ENU: {sample['heliostat_enu']}")
    print(f"  Motors: [{sample['motor_1']}, {sample['motor_2']}]")
    print(f"  Flux image shape: {sample['flux_image'].shape}")

Loading all calibration samples...

Loaded 3 samples:

Sample 0: AA31/126372
  Sun: elevation=50.56°, azimuth=59.79°
  Heliostat ENU: [-23.52598878  24.8462733    1.67156965]
  Motors: [27834, 20458]
  Flux image shape: (256, 256)

Sample 1: AA39/271633
  Sun: elevation=37.32°, azimuth=60.12°
  Heliostat ENU: [11.63168064 24.66215065  1.68888367]
  Motors: [30094, 33406]
  Flux image shape: (256, 256)

Sample 2: AC43/72752
  Sun: elevation=9.58°, azimuth=-64.90°
  Heliostat ENU: [29.31768332 33.76805692  1.73819322]
  Motors: [44706, 73338]
  Flux image shape: (256, 256)


In [6]:
# Train-test split: 2 train, 1 test
train_samples = all_samples[:2]  # AA31, AA39
test_samples = all_samples[2:]   # AC43

print(f"Training samples: {len(train_samples)}")
for sample in train_samples:
    print(f"  - {sample['heliostat_id']}/{sample['calibration_id']}")

print(f"\nTest samples: {len(test_samples)}")
for sample in test_samples:
    print(f"  - {sample['heliostat_id']}/{sample['calibration_id']}")

Training samples: 2
  - AA31/126372
  - AA39/271633

Test samples: 1
  - AC43/72752


## Prepare Dataset for Network

Now we'll prepare the data in the format needed for the neural network:
- **Inputs:** Sun angles (2) + Heliostat position (3) + Tower position (3) + Target position (3) + Flux image (256x256 flattened) = 65547 features
- **Outputs:** Motor positions (2)

## Make the dataset loadable for the training using torch

In [7]:
# Create PyTorch Dataset class with flux image as input
from torch.utils.data import Dataset, DataLoader

class HeliostatMotorDataset(Dataset):
    """Dataset for heliostat motor position prediction with flux image as input."""
    
    def __init__(self, samples_list):
        """
        Args:
            samples_list: List of sample dictionaries from load_sample_data()
        """
        self.samples = samples_list
        
        # Compute normalization statistics from the samples
        self._compute_normalization_stats()
        
    def _compute_normalization_stats(self):
        """Compute min/max for normalization."""
        # Sun angles - use reasonable bounds
        self.sun_elev_min, self.sun_elev_max = 0.0, 90.0  # degrees
        self.sun_azim_min, self.sun_azim_max = -180.0, 180.0  # degrees
        
        # Motor positions - compute from data
        all_motor_1 = [s['motor_1'] for s in self.samples]
        all_motor_2 = [s['motor_2'] for s in self.samples]
        self.motor_1_min, self.motor_1_max = min(all_motor_1), max(all_motor_1)
        self.motor_2_min, self.motor_2_max = min(all_motor_2), max(all_motor_2)
        
        # Position bounds - compute from data (ENU coordinates)
        all_positions = np.array([s['heliostat_enu'] for s in self.samples])
        self.pos_min = all_positions.min(axis=0)
        self.pos_max = all_positions.max(axis=0)
        
        print(f"Normalization statistics:")
        print(f"  Sun elevation: [{self.sun_elev_min}, {self.sun_elev_max}]")
        print(f"  Sun azimuth: [{self.sun_azim_min}, {self.sun_azim_max}]")
        print(f"  Motor 1: [{self.motor_1_min}, {self.motor_1_max}]")
        print(f"  Motor 2: [{self.motor_2_min}, {self.motor_2_max}]")
        print(f"  Position min: {self.pos_min}")
        print(f"  Position max: {self.pos_max}")
    
    def normalize(self, value, min_val, max_val):
        """Normalize value to [0, 1] range."""
        if max_val - min_val == 0:
            return 0.5
        return (value - min_val) / (max_val - min_val)
    
    def denormalize_motors(self, motor_1_norm, motor_2_norm):
        """Denormalize motor positions from [0, 1] back to increments."""
        motor_1 = motor_1_norm * (self.motor_1_max - self.motor_1_min) + self.motor_1_min
        motor_2 = motor_2_norm * (self.motor_2_max - self.motor_2_min) + self.motor_2_min
        return motor_1, motor_2
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        
        # Normalize geometric inputs
        sun_elev_norm = self.normalize(sample['sun_elevation'], self.sun_elev_min, self.sun_elev_max)
        sun_azim_norm = self.normalize(sample['sun_azimuth'], self.sun_azim_min, self.sun_azim_max)
        
        # Normalize heliostat position
        heliostat_pos_norm = np.array([
            self.normalize(sample['heliostat_enu'][0], self.pos_min[0], self.pos_max[0]),
            self.normalize(sample['heliostat_enu'][1], self.pos_min[1], self.pos_max[1]),
            self.normalize(sample['heliostat_enu'][2], self.pos_min[2], self.pos_max[2]),
        ])
        
        # Tower is at origin [0, 0, 0] - normalized to 0.5 (middle of range)
        tower_pos_norm = np.array([
            self.normalize(0.0, self.pos_min[0], self.pos_max[0]),
            self.normalize(0.0, self.pos_min[1], self.pos_max[1]),
            self.normalize(0.0, self.pos_min[2], self.pos_max[2]),
        ])
        
        # Normalize target position
        target_pos_norm = np.array([
            self.normalize(sample['target_enu'][0], self.pos_min[0], self.pos_max[0]),
            self.normalize(sample['target_enu'][1], self.pos_min[1], self.pos_max[1]),
            self.normalize(sample['target_enu'][2], self.pos_min[2], self.pos_max[2]),
        ])
        
        # Stack geometric features into a vector [11 features]
        geometric_features = np.concatenate([
            [sun_elev_norm, sun_azim_norm],  # 2
            heliostat_pos_norm,               # 3
            tower_pos_norm,                   # 3
            target_pos_norm,                  # 3
        ])
        
        # Flux image: already normalized to [0, 1] during loading
        # Flatten from [256, 256] to [65536]
        flux_flattened = sample['flux_image'].flatten()
        
        # Concatenate geometric features + flattened flux image
        # Total: 11 + 65536 = 65547 features
        full_inputs = np.concatenate([geometric_features, flux_flattened])
        
        # Normalize motor positions (outputs)
        motor_1_norm = self.normalize(sample['motor_1'], self.motor_1_min, self.motor_1_max)
        motor_2_norm = self.normalize(sample['motor_2'], self.motor_2_min, self.motor_2_max)
        outputs = np.array([motor_1_norm, motor_2_norm])
        
        # Convert to PyTorch tensors
        inputs_tensor = torch.tensor(full_inputs, dtype=torch.float32)
        outputs_tensor = torch.tensor(outputs, dtype=torch.float32)
        
        return {
            'inputs': inputs_tensor,           # [65547] = 11 geometric + 65536 flux pixels
            'motor_positions': outputs_tensor, # [2]
            'metadata': {
                'heliostat_id': sample['heliostat_id'],
                'calibration_id': sample['calibration_id'],
            }
        }

print("HeliostatMotorDataset class defined")


HeliostatMotorDataset class defined


In [8]:
# Create datasets for train and test splits
train_dataset = HeliostatMotorDataset(train_samples)
test_dataset = HeliostatMotorDataset(test_samples)

print(f"\nDataset created:")
print(f"  Training samples: {len(train_dataset)}")
print(f"  Test samples: {len(test_dataset)}")

# Test loading one sample
sample = train_dataset[0]
print(f"\nSample data shapes:")
print(f"  Inputs: {sample['inputs'].shape}")
print(f"  Motor positions: {sample['motor_positions'].shape}")
print(f"\nInput breakdown:")
print(f"  Geometric features (first 11): {sample['inputs'][:11]}")
print(f"  Flux pixels (65536 total, showing first 5): {sample['inputs'][11:16]}")
print(f"  Motor positions (normalized): {sample['motor_positions']}")


Normalization statistics:
  Sun elevation: [0.0, 90.0]
  Sun azimuth: [-180.0, 180.0]
  Motor 1: [27834, 30094]
  Motor 2: [20458, 33406]
  Position min: [-23.52598878  24.66215065   1.67156965]
  Position max: [11.63168064 24.8462733   1.68888367]
Normalization statistics:
  Sun elevation: [0.0, 90.0]
  Sun azimuth: [-180.0, 180.0]
  Motor 1: [44706, 44706]
  Motor 2: [73338, 73338]
  Position min: [29.31768332 33.76805692  1.73819322]
  Position max: [29.31768332 33.76805692  1.73819322]

Dataset created:
  Training samples: 2
  Test samples: 1

Sample data shapes:
  Inputs: torch.Size([65547])
  Motor positions: torch.Size([2])

Input breakdown:
  Geometric features (first 11): tensor([ 5.6173e-01,  6.6609e-01,  0.0000e+00,  1.0000e+00,  0.0000e+00,
         6.6916e-01, -1.3394e+02, -9.6544e+01,  6.6877e-01, -1.5152e+02,
         1.9759e+03])
  Flux pixels (65536 total, showing first 5): tensor([0., 0., 0., 0., 0.])
  Motor positions (normalized): tensor([0., 0.])


In [9]:
# Create DataLoaders
# Note: batch_size=1 since we only have 2 training samples
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

print("DataLoaders created")
print(f"  Train batches: {len(train_loader)}")
print(f"  Test batches: {len(test_loader)}")

# Test loading a batch
for batch in train_loader:
    print(f"\nBatch shapes:")
    print(f"  Inputs: {batch['inputs'].shape}")  # [batch_size, 65547]
    print(f"  Motor positions: {batch['motor_positions'].shape}")  # [batch_size, 2]
    break


DataLoaders created
  Train batches: 2
  Test batches: 1

Batch shapes:
  Inputs: torch.Size([1, 65547])
  Motor positions: torch.Size([1, 2])


## Define Neural Network

Simple feedforward network for motor position prediction:
- **Input:** 65,547 features (11 geometric + 65,536 flux pixels)
- **Hidden layers:** 256 → 128 → 64 neurons
- **Output:** 2 motor positions (normalized to [0, 1])
- **Activation:** ReLU for hidden layers, Sigmoid for output

In [10]:
import torch.nn as nn

class MotorPositionPredictor(nn.Module):
    """Simple feedforward neural network for motor position prediction."""
    
    def __init__(self, input_dim=65547, hidden_dim1=256, hidden_dim2=128, hidden_dim3=64, output_dim=2):
        super(MotorPositionPredictor, self).__init__()
        
        self.network = nn.Sequential(
            # Input layer → First hidden layer
            nn.Linear(input_dim, hidden_dim1),
            nn.ReLU(),
            
            # First hidden → Second hidden layer
            nn.Linear(hidden_dim1, hidden_dim2),
            nn.ReLU(),
            
            # Second hidden → Third hidden layer
            nn.Linear(hidden_dim2, hidden_dim3),
            nn.ReLU(),
            
            # Third hidden → Output layer
            nn.Linear(hidden_dim3, output_dim),
            nn.Sigmoid()  # Output in [0, 1] for normalized motor positions
        )
    
    def forward(self, x):
        """
        Forward pass through the network.
        
        Args:
            x: Input tensor of shape [batch_size, 65547]
            
        Returns:
            Predicted motor positions of shape [batch_size, 2]
        """
        return self.network(x)

print("MotorPositionPredictor network defined")

MotorPositionPredictor network defined


In [11]:
# Instantiate the network and move to device
model = MotorPositionPredictor().to(device)

# Print network architecture
print(f"\nNetwork architecture:")
print(model)

# Count total parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")


Network architecture:
MotorPositionPredictor(
  (network): Sequential(
    (0): Linear(in_features=65547, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): ReLU()
    (6): Linear(in_features=64, out_features=2, bias=True)
    (7): Sigmoid()
  )
)

Total parameters: 16,821,570
Trainable parameters: 16,821,570


In [12]:
# Test forward pass with one batch
test_batch = next(iter(train_loader))
test_input = test_batch['inputs'].to(device)
test_output = model(test_input)

print(f"\nTest forward pass:")
print(f"  Input shape: {test_input.shape}")
print(f"  Output shape: {test_output.shape}")
print(f"  Output values (normalized motor positions): {test_output.squeeze().detach().cpu().numpy()}")

# Denormalize to get actual motor positions
motor_1_norm, motor_2_norm = test_output.squeeze().detach().cpu().numpy()
motor_1_actual, motor_2_actual = train_dataset.denormalize_motors(motor_1_norm, motor_2_norm)
print(f"  Denormalized motor positions: [{motor_1_actual:.0f}, {motor_2_actual:.0f}] increments")


Test forward pass:
  Input shape: torch.Size([1, 65547])
  Output shape: torch.Size([1, 2])
  Output values (normalized motor positions): [0.42128184 0.41756198]
  Denormalized motor positions: [28786, 25865] increments


## Helper Functions for ARTIST Integration

Before training, we need helper functions to:
1. Convert sun angles to incident ray directions
2. Set up ARTIST scenario and raytracer

In [13]:
def sun_angles_to_incident_ray_direction(elevation_deg, azimuth_deg):
    """
    Convert sun elevation and azimuth angles to incident ray direction.
    
    Args:
        elevation_deg: Sun elevation angle in degrees (0° = horizon, 90° = zenith)
        azimuth_deg: Sun azimuth angle in degrees (0° = north, 90° = east, 180° = south, -90° = west)
        
    Returns:
        torch.Tensor: Incident ray direction [dx, dy, dz, 0] normalized
    """
    # Convert to radians
    elevation_rad = np.radians(elevation_deg)
    azimuth_rad = np.radians(azimuth_deg)
    
    # Convert spherical to Cartesian coordinates
    # The incident ray direction points FROM the sun TO the heliostat
    # In ENU coordinates:
    # - E (east) is x-axis
    # - N (north) is y-axis
    # - U (up) is z-axis
    
    # Compute direction components
    dx = np.sin(azimuth_rad) * np.cos(elevation_rad)  # East component
    dy = np.cos(azimuth_rad) * np.cos(elevation_rad)  # North component
    dz = -np.sin(elevation_rad)  # Up component (negative because rays come DOWN from sun)
    
    # Create direction vector [dx, dy, dz, 0]
    direction = np.array([dx, dy, dz, 0.0])
    
    # Normalize
    norm = np.sqrt(dx**2 + dy**2 + dz**2)
    direction[:3] = direction[:3] / norm
    
    return torch.tensor(direction, dtype=torch.float32)

# Test the function
test_elevation = 50.56  # degrees
test_azimuth = 59.79    # degrees
test_direction = sun_angles_to_incident_ray_direction(test_elevation, test_azimuth)
print(f"Sun angles: elevation={test_elevation}°, azimuth={test_azimuth}°")
print(f"Incident ray direction: {test_direction.numpy()}")
print(f"Direction magnitude (should be ~1.0): {torch.norm(test_direction[:3]).item():.6f}")

Sun angles: elevation=50.56°, azimuth=59.79°
Incident ray direction: [ 0.5489919   0.31964922 -0.7722903   0.        ]
Direction magnitude (should be ~1.0): 1.000000


## Generate ARTIST Scenario

First, we need to generate a scenario file using ARTIST. This needs to be done once for each heliostat configuration.

In [14]:
def generate_scenario_for_heliostat(
    heliostat_id: str,
    heliostat_properties_path: pathlib.Path,
    deflectometry_path: pathlib.Path,
    tower_measurements_path: pathlib.Path,
    output_path: pathlib.Path,
    device: torch.device
) -> None:
    """
    Generate an ARTIST scenario for a single heliostat.
    
    Args:
        heliostat_id: Name/ID of the heliostat (e.g., 'AA31')
        heliostat_properties_path: Path to heliostat-properties.json
        deflectometry_path: Path to deflectometry.h5
        tower_measurements_path: Path to tower-measurements.json
        output_path: Path where scenario.h5 will be saved
        device: Torch device to use for computation
    """
    from artist.data_parser import paint_scenario_parser
    from artist.scenario.configuration_classes import LightSourceConfig, LightSourceListConfig
    from artist.scenario.h5_scenario_generator import H5ScenarioGenerator
    
    print(f"Generating scenario for heliostat {heliostat_id}...")
    
    # Configure light source
    light_source_config = LightSourceConfig(
        light_source_key="sun_1",
        light_source_type=config_dictionary.sun_key,
        number_of_rays=10,
        distribution_type=config_dictionary.light_source_distribution_is_normal,
        mean=0.0,
        covariance=4.3681e-06,
    )
    
    light_source_list_config = LightSourceListConfig(light_source_list=[light_source_config])
    
    # Extract tower measurements
    power_plant_config, target_area_list_config = paint_scenario_parser.extract_paint_tower_measurements(
        tower_measurements_path=tower_measurements_path,
        device=device
    )
    
    # Prepare heliostat files
    heliostat_files_list = [
        (
            heliostat_id,
            heliostat_properties_path,
            deflectometry_path,
        ),
    ]
    
    # Extract heliostat configuration with NURBS surface fitting
    number_of_nurbs_control_points = torch.tensor([20, 20], device=device)
    nurbs_fit_optimizer = torch.optim.Adam([torch.empty(1, requires_grad=True)], lr=1e-3)
    nurbs_fit_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        nurbs_fit_optimizer, mode="min", factor=0.2, patience=50, threshold=1e-7, threshold_mode="abs"
    )
    
    print("  Fitting NURBS surface from deflectometry data...")
    heliostat_list_config, prototype_config = paint_scenario_parser.extract_paint_heliostats_fitted_surface(
        paths=heliostat_files_list,
        power_plant_position=power_plant_config.power_plant_position,
        number_of_nurbs_control_points=number_of_nurbs_control_points,
        deflectometry_step_size=100,
        nurbs_fit_method=config_dictionary.fit_nurbs_from_normals,
        nurbs_fit_tolerance=1e-10,
        nurbs_fit_max_epoch=400,
        nurbs_fit_optimizer=nurbs_fit_optimizer,
        nurbs_fit_scheduler=nurbs_fit_scheduler,
        device=device,
    )
    
    # Generate scenario
    print("  Generating scenario file...")
    scenario_generator = H5ScenarioGenerator(
        file_path=output_path,
        power_plant_config=power_plant_config,
        target_area_list_config=target_area_list_config,
        light_source_list_config=light_source_list_config,
        prototype_config=prototype_config,
        heliostat_list_config=heliostat_list_config,
    )
    scenario_generator.generate_scenario()
    
    print(f"✓ Scenario generated successfully: {output_path}")

print("Scenario generation function defined")

Scenario generation function defined


In [15]:
# For simplicity, we'll use ONE heliostat scenario for initial training
# In the future, you can create separate scenarios for each heliostat or combine them

# We'll use AA31 as the example heliostat
for sample in CALIBRATION_SAMPLES:
    sample['heliostat_id']

    HELIOSTAT_ID = sample['heliostat_id']
    SCENARIO_DIR = pathlib.Path("scenarios")
    SCENARIO_DIR.mkdir(parents=True, exist_ok=True)

    scenario_path = SCENARIO_DIR / f"scenario_{HELIOSTAT_ID}.h5"

    print(f"Scenario path: {scenario_path}")
    print(f"Scenario exists: {scenario_path.exists()}")

    if not scenario_path.exists():
        print(f"\n⚠️ Sample {HELIOSTAT_ID} is missing a scenario file. Generating scenario...")
        
        # Generate the scenario using the function
        generate_scenario_for_heliostat(
            heliostat_id=HELIOSTAT_ID,
            heliostat_properties_path=DATA_DIR / HELIOSTAT_ID / "heliostat-properties.json",
            deflectometry_path=DATA_DIR / HELIOSTAT_ID / "deflectometry.h5",
            tower_measurements_path=TOWER_FILE,
            output_path=scenario_path,
            device=device
        )
    else:
        print("✓ Scenario file found!")

Scenario path: scenarios/scenario_AA31.h5
Scenario exists: True
✓ Scenario file found!
Scenario path: scenarios/scenario_AA39.h5
Scenario exists: True
✓ Scenario file found!
Scenario path: scenarios/scenario_AC43.h5
Scenario exists: True
✓ Scenario file found!


## Training Configuration and Setup

Define training parameters and set up the optimizer.

In [16]:
# Training parameters
LEARNING_RATE = 1e-3
NUM_EPOCHS = 100
LOG_INTERVAL = 5

# Create optimizer and loss function
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
loss_fn = torch.nn.MSELoss()  # Simple MSE loss for flux images

# You can also use ARTIST's loss functions:
# from artist.core.loss_functions import KLDivergenceLoss
# loss_fn = KLDivergenceLoss()

print(f"Training configuration:")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Number of epochs: {NUM_EPOCHS}")
print(f"  Loss function: {loss_fn.__class__.__name__}")
print(f"  Optimizer: {optimizer.__class__.__name__}")

Training configuration:
  Learning rate: 0.001
  Number of epochs: 100
  Loss function: MSELoss
  Optimizer: Adam


## Training Loop with ARTIST Integration

**Note:** This training loop assumes the scenario file exists. For the first run, you'll train with just motor position loss. Once you have the scenario file, you can enable flux-based loss with ARTIST raytracing.

In [17]:
# Check if we can use ARTIST (scenario file exists)
USE_ARTIST = scenario_path.exists()
NUMBER_OF_RAYS = 10
if USE_ARTIST:
    print("✓ ARTIST scenario found - will use flux-based loss with raytracing")
    print("Loading scenario...")
    
    # Load ARTIST scenario
    with h5py.File(scenario_path, 'r') as scenario_file:
        scenario = Scenario.load_scenario_from_hdf5(
            scenario_file=scenario_file,
            device=device
        )
    
    scenario.set_number_of_rays(number_of_rays=NUMBER_OF_RAYS)  # Adjust based on your GPU memory
    
    # Get the heliostat group (assuming single heliostat scenario)
    heliostat_group = scenario.heliostat_field.heliostat_groups[0]
    
    # Define target area index (check scenario for correct index)
    # For solar_tower_juelich_lower, you may need to inspect the scenario
    target_area_index = 2  # Adjust this based on your scenario
    
    print(f"  Scenario loaded successfully")
    print(f"  Number of heliostats in group: {heliostat_group.number_of_heliostats}")
    print(f"  Number of rays: {NUMBER_OF_RAYS}")
else:
    print("⚠️ ARTIST scenario NOT found - will use motor position loss only")
    print("This is a simplified training mode without raytracing")

✓ ARTIST scenario found - will use flux-based loss with raytracing
Loading scenario...
[2025-12-15 16:24:34,188][artist.scenario.scenario][INFO] - Loading an ARTIST scenario HDF5 file. This scenario file is version 1.0.
[2025-12-15 16:24:34,189][artist.field.tower_target_areas][INFO] - Loading the tower target areas from an HDF5 file.
[2025-12-15 16:24:34,190][artist.field.tower_target_areas][WARNING] - No curvature in the east direction set for the multi_focus_tower!
[2025-12-15 16:24:34,190][artist.field.tower_target_areas][WARNING] - No curvature in the up direction set for the multi_focus_tower!
[2025-12-15 16:24:34,191][artist.field.tower_target_areas][WARNING] - No curvature in the east direction set for the receiver!
[2025-12-15 16:24:34,191][artist.field.tower_target_areas][WARNING] - No curvature in the up direction set for the receiver!
[2025-12-15 16:24:34,192][artist.field.tower_target_areas][WARNING] - No curvature in the east direction set for the solar_tower_juelich_lowe

In [18]:
# Training loop
train_losses = []
test_losses = []

print("\n" + "="*80)
print("STARTING TRAINING")
print("="*80)

for epoch in range(NUM_EPOCHS):
    model.train()
    epoch_loss = 0.0
    num_batches = 0
    
    for batch_idx, batch in enumerate(train_loader):
        optimizer.zero_grad()
        
        # 1. Network forward pass
        inputs = batch['inputs'].to(device)
        ground_truth_motors = batch['motor_positions'].to(device)
        
        predicted_motors_normalized = model(inputs)  # [batch_size, 2] in [0, 1]
        
        # 2. Denormalize motor positions (preserving gradients!)
        motor_1_actual = (
            predicted_motors_normalized[:, 0] * (train_dataset.motor_1_max - train_dataset.motor_1_min) 
            + train_dataset.motor_1_min
        )
        motor_2_actual = (
            predicted_motors_normalized[:, 1] * (train_dataset.motor_2_max - train_dataset.motor_2_min) 
            + train_dataset.motor_2_min
        )
        predicted_motors_actual = torch.stack([motor_1_actual, motor_2_actual], dim=1)  # [batch_size, 2]
        
        if USE_ARTIST:
            # ============================================
            # ARTIST INTEGRATION: Flux-based training
            # ============================================
            
            # Get sample metadata
            sample = train_samples[batch_idx]
            
            # Convert sun angles to incident ray direction
            incident_ray_direction = sun_angles_to_incident_ray_direction(
                sample['sun_elevation'],
                sample['sun_azimuth']
            ).unsqueeze(0).to(device)  # [1, 4]
            
            # Active heliostats mask (single heliostat)
            active_heliostats_mask = torch.tensor([1], dtype=torch.int32, device=device)
            
            # Activate heliostats
            heliostat_group.activate_heliostats(
                active_heliostats_mask=active_heliostats_mask,
                device=device
            )
            
            # Align surfaces with predicted motor positions
            heliostat_group.align_surfaces_with_motor_positions(
                motor_positions=predicted_motors_actual,  # From network!
                active_heliostats_mask=active_heliostats_mask,
                device=device
            )
            
            # Create ray tracer
            ray_tracer = HeliostatRayTracer(
                scenario=scenario,
                heliostat_group=heliostat_group,
                batch_size=1,
                bitmap_resolution=torch.tensor([256, 256], device=device)
            )
            
            # Trace rays to get predicted flux
            target_area_mask = torch.tensor([target_area_index], device=device)
            predicted_flux = ray_tracer.trace_rays(
                incident_ray_directions=incident_ray_direction,
                active_heliostats_mask=active_heliostats_mask,
                target_area_mask=target_area_mask,
                device=device
            )  # [1, 256, 256]
            
            # Get ground truth flux from dataset
            ground_truth_flux = torch.tensor(sample['flux_image'], device=device).unsqueeze(0)  # [1, 256, 256]
            
            # Compute flux-based loss
            flux_loss = loss_fn(predicted_flux, ground_truth_flux)
            
            # Optional: Add motor position regularization
            motor_loss = loss_fn(predicted_motors_normalized, ground_truth_motors) * 0.1
            
            loss = flux_loss + motor_loss
            
        else:
            # ============================================
            # SIMPLIFIED MODE: Motor position loss only
            # ============================================
            loss = loss_fn(predicted_motors_normalized, ground_truth_motors)
        
        # 3. Backpropagation
        loss.backward()
        
        # 4. Optimizer step
        optimizer.step()
        
        epoch_loss += loss.item()
        num_batches += 1
    
    # Calculate average loss for the epoch
    avg_train_loss = epoch_loss / num_batches
    train_losses.append(avg_train_loss)
    
    # Validation on test set
    model.eval()
    test_loss = 0.0
    with torch.no_grad():
        for batch_idx, batch in enumerate(test_loader):
            inputs = batch['inputs'].to(device)
            ground_truth_motors = batch['motor_positions'].to(device)
            
            predicted_motors_normalized = model(inputs)
            
            if not USE_ARTIST:
                # Simple motor position loss
                loss = loss_fn(predicted_motors_normalized, ground_truth_motors)
            else:
                # Denormalize
                motor_1_actual = (
                    predicted_motors_normalized[:, 0] * (test_dataset.motor_1_max - test_dataset.motor_1_min) 
                    + test_dataset.motor_1_min
                )
                motor_2_actual = (
                    predicted_motors_normalized[:, 1] * (test_dataset.motor_2_max - test_dataset.motor_2_min) 
                    + test_dataset.motor_2_min
                )
                predicted_motors_actual = torch.stack([motor_1_actual, motor_2_actual], dim=1)
                
                # For validation, just use motor loss (faster than raytracing)
                loss = loss_fn(predicted_motors_normalized, ground_truth_motors)
            
            test_loss += loss.item()
    
    avg_test_loss = test_loss / len(test_loader)
    test_losses.append(avg_test_loss)
    
    # Logging
    if epoch % LOG_INTERVAL == 0 or epoch == NUM_EPOCHS - 1:
        print(f"Epoch {epoch:3d}/{NUM_EPOCHS} | Train Loss: {avg_train_loss:.6f} | Test Loss: {avg_test_loss:.6f}")

print("\n" + "="*80)
print("TRAINING COMPLETE")
print("="*80)


STARTING TRAINING


ValueError: No ray intersections on the front of the target area planes.

## Visualize Training Progress

In [ ]:
# Plot training and test losses
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Train Loss', marker='o', markersize=3)
plt.plot(test_losses, label='Test Loss', marker='s', markersize=3)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Progress')
plt.legend()
plt.grid(True, alpha=0.3)
plt.yscale('log')  # Log scale often helps visualize loss curves
plt.tight_layout()
plt.savefig('outputs/training_loss.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nFinal Train Loss: {train_losses[-1]:.6f}")
print(f"Final Test Loss: {test_losses[-1]:.6f}")

## Evaluate Predictions

Let's evaluate the trained model on a test sample and compare predicted vs actual motor positions.

In [ ]:
# Evaluate on test set
model.eval()

test_sample = test_dataset[0]
test_input = test_sample['inputs'].unsqueeze(0).to(device)
ground_truth_motors_norm = test_sample['motor_positions'].cpu().numpy()

# Predict
with torch.no_grad():
    predicted_motors_norm = model(test_input).squeeze().cpu().numpy()

# Denormalize
pred_motor_1, pred_motor_2 = test_dataset.denormalize_motors(
    predicted_motors_norm[0],
    predicted_motors_norm[1]
)

gt_motor_1, gt_motor_2 = test_dataset.denormalize_motors(
    ground_truth_motors_norm[0],
    ground_truth_motors_norm[1]
)

print("="*80)
print("TEST SAMPLE EVALUATION")
print("="*80)
print(f"\nHeliostat: {test_sample['metadata']['heliostat_id']}")
print(f"Calibration: {test_sample['metadata']['calibration_id']}")

print(f"\nGround Truth Motor Positions:")
print(f"  Motor 1: {gt_motor_1:.0f} increments")
print(f"  Motor 2: {gt_motor_2:.0f} increments")

print(f"\nPredicted Motor Positions:")
print(f"  Motor 1: {pred_motor_1:.0f} increments (error: {pred_motor_1 - gt_motor_1:.0f})")
print(f"  Motor 2: {pred_motor_2:.0f} increments (error: {pred_motor_2 - gt_motor_2:.0f})")

print(f"\nRelative Errors:")
print(f"  Motor 1: {abs(pred_motor_1 - gt_motor_1) / gt_motor_1 * 100:.2f}%")
print(f"  Motor 2: {abs(pred_motor_2 - gt_motor_2) / gt_motor_2 * 100:.2f}%")

## Save the Trained Model

In [ ]:
# Create outputs directory
outputs_dir = pathlib.Path("outputs")
outputs_dir.mkdir(exist_ok=True)

# Save model checkpoint
model_path = outputs_dir / "motor_predictor_model.pt"
torch.save({
    'epoch': NUM_EPOCHS,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'train_losses': train_losses,
    'test_losses': test_losses,
    'normalization_stats': {
        'motor_1_min': train_dataset.motor_1_min,
        'motor_1_max': train_dataset.motor_1_max,
        'motor_2_min': train_dataset.motor_2_min,
        'motor_2_max': train_dataset.motor_2_max,
    }
}, model_path)

print(f"✓ Model saved to: {model_path}")
print(f"  File size: {model_path.stat().st_size / 1024 / 1024:.2f} MB")

## Summary and Next Steps

**What you've accomplished:**
1. ✓ Loaded and preprocessed PAINT calibration data
2. ✓ Created PyTorch Dataset with flux images as inputs
3. ✓ Defined a neural network for motor position prediction
4. ✓ Implemented training loop with ARTIST integration (if scenario exists)
5. ✓ Trained the model and evaluated predictions

**Current Training Mode:**
- **Without ARTIST scenario:** Trains on motor position loss only
- **With ARTIST scenario:** Trains with differentiable flux-based loss

**Next steps to improve the pipeline:**

1. **Generate ARTIST Scenario:**
   - Run the scenario generation code (shown in cell above)
   - This enables flux-based training with raytracing

2. **Expand Dataset:**
   - Use all available calibration measurements (currently using only 3)
   - Generate synthetic training data with different sun angles

3. **Improve Network Architecture:**
   - Try convolutional layers for flux image processing
   - Experiment with separate encoders for geometric features and flux
   - Add residual connections or attention mechanisms

4. **Advanced Loss Functions:**
   - Use ARTIST's KLDivergenceLoss for distribution matching
   - Add regularization terms (motor position smoothness, etc.)
   - Weighted combination of flux loss + motor position loss

5. **Multi-Heliostat Training:**
   - Create scenarios for all 3 heliostats
   - Train a single network that works for multiple heliostats
   - Add heliostat ID as one-hot encoded input feature

6. **Hyperparameter Tuning:**
   - Experiment with learning rates, batch sizes, network sizes
   - Add learning rate scheduling
   - Implement early stopping

7. **Validation and Visualization:**
   - Compare predicted vs ground truth flux images
   - Visualize motor position predictions over time
   - Analyze failure cases